# GIGABYTE AORUS MASTER 16 AM6H — RAG Demo (Colab)

**Menu → Runtime → Change runtime type → T4 GPU** 先開 GPU。

Pipeline：
1. 安裝環境（uv + llama-cpp-python CUDA build）
2. 爬取規格資料 → 分塊 → 建向量索引
3. 互動問答（含 TTFT / TPS 量測）
4. Benchmark 評測

## Step 1：安裝 uv 與 clone repo

In [ ]:
# 安裝 uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
!uv --version

In [ ]:
# Clone repo（已有則 pull）
import os

REPO_URL = "https://github.com/chiahua-yang/GIGABYTE-RAG-test.git"
BRANCH   = "claude/build-rag-product-qa-2nMZM"
REPO_DIR = "/content/GIGABYTE-RAG-test"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

## Step 2：安裝依賴

`llama-cpp-python` 需要用 CUDA 旗標重新編譯，否則只用 CPU 推論（速度慢 10x）。

In [ ]:
# 檢查 GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# 安裝其他依賴（不含 llama-cpp-python，下面單獨裝）
!uv sync --no-install-package llama-cpp-python

In [ ]:
# 安裝 llama-cpp-python with CUDA support
# 如果 Colab 的 CUDA 版本是 12.x 用下面這行
!CMAKE_ARGS="-DGGML_CUDA=on" uv pip install llama-cpp-python --force-reinstall --no-cache-dir

# 驗證
!uv run python -c "from llama_cpp import Llama; print('llama-cpp-python OK')" 2>&1 | tail -1

## Step 3：爬取規格資料

In [ ]:
# 先確認 requests 能拿到有效的 HTML
import requests

URL = "https://www.gigabyte.com/tw/Laptop/AORUS-MASTER-16-AM6H/sp"
headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36"
}
resp = requests.get(URL, headers=headers, timeout=30)
print(f"Status: {resp.status_code}  |  Content length: {len(resp.text):,} bytes")

# 簡單檢查是否包含規格關鍵字
for keyword in ["RTX", "DDR5", "CPU", "Ryzen", "顯示"]:
    found = keyword in resp.text
    print(f"  '{keyword}': {'✓' if found else '✗ (頁面可能是 JS 渲染)'}")

In [ ]:
# 如果上面出現 ✗，代表頁面是 JS 渲染，改用 playwright
# 如果都是 ✓，跳過這格直接執行下一格

import subprocess
result = subprocess.run(["uv", "run", "python", "-c", "import requests; r=requests.get('https://www.gigabyte.com/tw/Laptop/AORUS-MASTER-16-AM6H/sp'); print(('RTX' in r.text))"], capture_output=True, text=True)
needs_playwright = result.stdout.strip() == "False"

if needs_playwright:
    print("頁面是 JS 渲染，安裝 playwright ...")
    !uv add playwright
    !uv run playwright install chromium
    !uv run playwright install-deps chromium
else:
    print("requests 可直接爬取，不需要 playwright")

In [ ]:
# 如果需要 playwright，先更新 scraper.py
# （這格只在 needs_playwright=True 時才需要執行）

PLAYWRIGHT_PATCH = '''
def fetch_html(url: str) -> str:
    """Playwright fallback for JS-rendered pages."""
    from playwright.sync_api import sync_playwright
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(url, wait_until="networkidle", timeout=60000)
        # Wait for spec table to load
        try:
            page.wait_for_selector(".spec", timeout=15000)
        except Exception:
            pass
        html = page.content()
        browser.close()
    return html
'''

if needs_playwright:
    # Patch the scraper
    scraper_path = "src/data/scraper.py"
    with open(scraper_path) as f:
        content = f.read()

    # Replace the fetch_html function
    import re
    content = re.sub(
        r'def fetch_html\(url: str\) -> str:.*?return resp\.text',
        PLAYWRIGHT_PATCH.strip(),
        content,
        flags=re.DOTALL
    )
    with open(scraper_path, "w") as f:
        f.write(content)
    print("scraper.py patched to use playwright")
else:
    print("No patch needed")

In [ ]:
# 執行爬取
!mkdir -p data
!uv run scrape

In [ ]:
# 確認資料
import json

with open("data/specs.json") as f:
    specs = json.load(f)

print(f"Total records: {len(specs)}")
models = list({r['model'] for r in specs})
print(f"Models found: {models}")

if len(specs) == 0:
    print("\n⚠️ 沒有解析到資料！請檢查 scraper.py 的 parser 邏輯")
else:
    print("\n前 5 筆:")
    for r in specs[:5]:
        print(f"  {r}")

## Step 4：建立向量索引

Embedding model `BAAI/bge-small-zh-v1.5` 會自動下載（~93MB），執行在 CPU 上。

In [ ]:
# Chunk + Index
!uv run python -m src.data.chunker
!uv run index

In [ ]:
# 確認 chunks
with open("data/chunks.json") as f:
    chunks = json.load(f)

print(f"Total chunks: {len(chunks)}")
for c in chunks:
    print(f"  [{c['model']:35s}] {c['category']}")

## Step 5：下載 LLM

**Qwen2.5-3B-Instruct-Q4_K_M** (~2.0 GB)  
在 4GB VRAM 預算下：LLM ~2.0GB + KV Cache ~0.8GB = ~3.0GB（有餘裕）

In [ ]:
!mkdir -p models

# 下載（約 2GB，需要幾分鐘）
!uv run python -c "
from src.rag.generator import download_model
path = download_model()
print('Model downloaded to:', path)
"

## Step 6：RAG 問答（含 TTFT / TPS）

In [ ]:
# 載入所有元件
import sys
sys.path.insert(0, "/content/GIGABYTE-RAG-test")

from sentence_transformers import SentenceTransformer
from src.rag.indexer import load_index, retrieve, EMBED_MODEL
from src.rag.generator import load_model, build_prompt, stream_generate

print("Loading index ...")
index = load_index()

print("Loading embedding model ...")
embed_model = SentenceTransformer(EMBED_MODEL, device="cpu")

print("Loading LLM ...")
llm = load_model(n_gpu_layers=-1)  # -1 = 全部 offload 到 GPU

print("\nAll components loaded!")

In [ ]:
def ask(question: str, top_k: int = 3):
    """Single-shot RAG query with metrics."""
    print(f"Q: {question}")
    print("-" * 60)

    # Retrieve
    chunks = retrieve(question, index, embed_model, top_k=top_k)
    print(f"[Retrieved chunks]")
    for c in chunks:
        print(f"  {c['id']}  (score={c['score']:.4f})")
    print()

    # Generate
    messages = build_prompt(question, chunks)
    print("[Answer]")
    answer = ""
    metrics = {}
    for token, m in stream_generate(llm, messages):
        print(token, end="", flush=True)
        answer += token
        if m:
            metrics = m

    print(f"\n\n[Metrics] TTFT={metrics.get('ttft')}s | TPS={metrics.get('tps')} tok/s | tokens={metrics.get('total_tokens')}")
    return answer, metrics

In [ ]:
# 測試問題 1：直接查找
ask("這台筆電的 CPU 是什麼型號？")

In [ ]:
# 測試問題 2：型號指定
ask("AM6H-BZH 的顯卡是什麼？VRAM 有多少？")

In [ ]:
# 測試問題 3：跨型號比較
ask("三個型號 BZH、BYH、BXH 的 GPU 差異是什麼？")

In [ ]:
# 測試問題 4：英文
ask("What GPU does the AM6H-BYH have and how much VRAM?")

## Step 7：完整 Benchmark 評測（20 題）

In [ ]:
!uv run eval --top-k 3 --max-tokens 256 --output data/benchmark_results.json

In [ ]:
# 顯示 benchmark summary
with open("data/benchmark_results.json") as f:
    results = json.load(f)

print("=" * 50)
print("BENCHMARK SUMMARY")
print("=" * 50)
for k, v in results["summary"].items():
    print(f"  {k}: {v}")

print("\n=" * 50)
print("PER-QUESTION RESULTS")
print("=" * 50)
for r in results["results"]:
    status = "✓" if r["retrieval_recall"] else "✗"
    print(f"  [{r['id']}] {status} recall | kw={r['keyword_hit_rate']:.0%} | ttft={r['metrics'].get('ttft','?')}s | tps={r['metrics'].get('tps','?')}")

## Step 8：VRAM 使用確認

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv